# Solutions — Series, DataFrame, read_csv / read_json, Profiling, Chunks

Exercise notebook: `09_pandas/01_dataframe_basics_io.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Load with explicit dtypes

**Working solution**

In [ ]:
import pandas as pd

def load_orders(path):
    df = pd.read_csv(path, dtype={"order_id": str, "customer_id": str, "product_id": str})
    return df.assign(
        quantity=pd.to_numeric(df["quantity"], errors="coerce"),
        unit_price=pd.to_numeric(df["unit_price"], errors="coerce"),
        order_ts=pd.to_datetime(df["order_ts"], utc=True, format="ISO8601", errors="coerce"),
        updated_at=pd.to_datetime(df["updated_at"], utc=True, format="ISO8601", errors="coerce"),
    )

**Explanation**

Let pandas guess and an id like `00123` becomes the integer `123`. `errors="coerce"` turns bad values into NaN/NaT so you can count and reject them in a vectorized way.
`.assign(...)` returns a new DataFrame — the idiomatic, chainable way to add or replace columns.

**Complexity:** O(n), vectorized.

**Common mistakes**

- Leaving dates as strings and comparing them later.
- `errors='ignore'` (deprecated) — keeps the whole column as object.

## Exercise 2 — 🟢 Easy · Series statistics

**Working solution**

In [ ]:
def series_stats(s):
    valid = s.dropna()
    if valid.empty:
        return {"count": 0, "nulls": int(s.isna().sum()), "mean": None, "median": None, "p95": None, "min": None, "max": None}
    return {"count": int(valid.count()), "nulls": int(s.isna().sum()), "mean": round(float(valid.mean()), 2),
            "median": round(float(valid.median()), 2), "p95": round(float(valid.quantile(0.95)), 2),
            "min": round(float(valid.min()), 2), "max": round(float(valid.max()), 2)}

**Explanation**

Series methods are vectorized reductions. Convert numpy scalars to plain `int`/`float` before returning to non-pandas code (JSON serialization, comparisons).

**Complexity:** O(n) (median/quantile O(n log n)).

**Common mistakes**

- Returning `numpy.int64` values that break `json.dumps`.

## Exercise 3 — 🟡 Medium · Flatten nested JSON with json_normalize

**Working solution**

In [ ]:
import json
import pandas as pd

def load_order_items(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    df = pd.json_normalize(data, record_path="items",
                           meta=["id", "created_at", "status", ["customer", "id"], ["customer", "country"], ["payment", "status"]],
                           meta_prefix="order.", errors="ignore")
    df = df.rename(columns={"order.id": "order_id", "order.created_at": "created_at", "order.status": "status",
                            "order.customer.id": "customer_id", "order.customer.country": "country",
                            "order.payment.status": "payment_status"})
    df["line_total"] = df["qty"] * df["price"]
    return df[["order_id", "created_at", "status", "customer_id", "country", "payment_status",
               "product_id", "name", "category", "qty", "price", "line_total"]]

**Explanation**

`json_normalize` with `record_path` explodes a nested list and `meta` copies parent fields onto each child row — Spark's `explode` + `select('parent.*')`. `meta_prefix` avoids name clashes (items and orders both have fields).

**Complexity:** O(total items).

**Common mistakes**

- Looping over orders and appending rows to a DataFrame one at a time (quadratic).

## Exercise 4 — 🟠 Hard · Profile a DataFrame

**Working solution**

In [ ]:
import pandas as pd

def profile(df):
    rows = {}
    for col in df.columns:
        s = df[col]
        non_null = s.dropna()
        if pd.api.types.is_string_dtype(s) or s.dtype == object:
            empty = int(non_null.astype(str).str.strip().eq("").sum())
        else:
            empty = 0
        rows[col] = {"dtype": str(s.dtype), "non_null": int(s.notna().sum()), "nulls": int(s.isna().sum()),
                     "null_pct": round(float(s.isna().mean() * 100), 2) if len(s) else 0.0,
                     "distinct": int(non_null.nunique()), "empty_strings": empty,
                     "example": str(non_null.iloc[0]) if len(non_null) else None}
    return pd.DataFrame.from_dict(rows, orient="index")

**Explanation**

Looping over **columns** is fine (there are few); looping over **rows** is what to avoid. `s.isna().mean()` is the null ratio in one vectorized call (booleans average to a fraction).

**Complexity:** O(n × columns).

**Common mistakes**

- `df.iterrows()` for profiling — 100× slower.
- Treating `''` as non-missing without reporting it — CSVs encode missing values both ways.

## Challenge — 🔴 Very Hard · Process a file in chunks

**Working solution**

In [ ]:
import pandas as pd

def revenue_by_country_chunked(path, chunksize=1000):
    partials = []
    for chunk in pd.read_csv(path, dtype=str, chunksize=chunksize):
        qty = pd.to_numeric(chunk["quantity"], errors="coerce")
        price = pd.to_numeric(chunk["unit_price"], errors="coerce")
        valid = chunk.assign(quantity=qty, unit_price=price)[(qty >= 1) & (price >= 0)].drop_duplicates()
        partials.append((valid["quantity"] * valid["unit_price"]).groupby(valid["country"]).sum())
    if not partials:
        return pd.Series(dtype=float, name="revenue")
    total = pd.concat(partials).groupby(level=0).sum().round(2).sort_values(ascending=False)
    total.index.name = "country"
    return total.rename("revenue")

**Explanation**

Sums are *decomposable*: the sum of per-chunk sums equals the global sum. Means are not (combine sums and counts instead), distinct counts and medians aren't either.
That decomposability is exactly what makes Spark's partial (map-side) aggregation work.

**Complexity:** O(n) time, O(chunksize) memory.

**Common mistakes**

- Averaging per-chunk averages.
- Deduplicating per chunk and believing duplicates across chunks are gone.